In [1]:
import time
import random
import pandas as pd

from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager



districts = [
    "Bagalkot",
    "Ballari",
    "Belagavi",
    "Bengaluru Rural",
    "Bengaluru Urban",
    "Bidar",
    "Chamarajanagar",
    "Chikkaballapur",
    "Chikkamagaluru",
    "Chitradurga",
    "Dakshina Kannada",
    "Davanagere",
    "Dharwad",
    "Gadag",
    "Hassan",
    "Haveri",
    "Kalaburagi",
    "Kodagu",
    "Kolar",
    "Koppal",
    "Mandya",
    "Mysuru",
    "Raichur",
    "Ramanagara",
    "Shivamogga",
    "Tumakuru",
    "Udupi",
    "Uttara Kannada",
    "Vijayapura",
    "Yadgir"
]

search_types = [
    "hospitals"
]


options = webdriver.ChromeOptions()

options.add_argument("--start-maximized")

options.add_argument(
    "--disable-blink-features=AutomationControlled"
)

options.add_argument("--disable-notifications")


driver = webdriver.Chrome(

    service=Service(
        ChromeDriverManager().install()
    ),

    options=options
)


driver.get("https://www.google.com/maps")

print("Opening Google Maps...")

time.sleep(15)

print("Google Maps Loaded")


all_data = []

visited_links = set()


def get_search_box():

    selectors = [

        '//input[@id="searchboxinput"]',
        '//input[contains(@placeholder,"Search")]',
        '//input[contains(@aria-label,"Search")]',
        '//input'

    ]

    for selector in selectors:

        try:

            box = driver.find_element(
                By.XPATH,
                selector
            )

            return box

        except:
            pass

    return None


def extract_school_data():

    try:

        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            name = ""

        try:
            address = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"address")]'
            ).text
        except:
            address = ""

        try:
            phone = driver.find_element(
                By.XPATH,
                '//button[contains(@data-item-id,"phone")]'
            ).text
        except:
            phone = ""

        try:
            website = driver.find_element(
                By.XPATH,
                '//a[contains(@data-item-id,"authority")]'
            ).get_attribute("href")
        except:
            website = ""

        try:
            rating = driver.find_element(
                By.XPATH,
                '//div[@role="img"]'
            ).get_attribute("aria-label")
        except:
            rating = ""

        return {

            "Hospital Name": name,
            "Address": address,
            "Phone": phone,
            "Website": website,
            "Rating": rating,
            "Google Maps URL": driver.current_url

        }

    except:

        return None


for district in districts:

    for search_type in search_types:

        try:

            query = (
                f"{search_type} "
                f"in {district} Karnataka"
            )

            print(f"\nSearching: {query}")


            search_box = get_search_box()

            if not search_box:

                print("Search Box Not Found")

                continue

            ActionChains(driver).move_to_element(
                search_box
            ).click().perform()

            time.sleep(1)

            search_box.send_keys(
                Keys.CONTROL + "a"
            )

            time.sleep(1)

            search_box.send_keys(Keys.DELETE)

            time.sleep(1)

            for char in query:

                search_box.send_keys(char)

                time.sleep(0.03)

            search_box.send_keys(Keys.ENTER)

            print("Search Submitted")

            time.sleep(8)


            try:

                scrollable_div = driver.find_element(
                    By.XPATH,
                    '//div[@role="feed"]'
                )

                previous_count = 0

                stagnant = 0

                for i in range(100):

                    driver.execute_script(
                        """
                        arguments[0].scrollTop =
                        arguments[0].scrollHeight
                        """,
                        scrollable_div
                    )

        
                    time.sleep(4)

                    listings = driver.find_elements(
                        By.XPATH,
                        '//a[contains(@href,"/maps/place/")]'
                    )

                    current_count = len(listings)

                    print(
                        f"{district} | "
                        f"{search_type} | "
                        f"Scroll {i+1} -> "
                        f"{current_count}"
                    )

                    if current_count == previous_count:

                        stagnant += 1

                    else:

                        stagnant = 0

                    if stagnant >= 4:

                        break

                    previous_count = current_count

            except Exception as e:

                print(f"Scrolling Error: {e}")


            listings = driver.find_elements(
                By.XPATH,
                '//a[contains(@href,"/maps/place/")]'
            )

            links = []

            for listing in listings:

                try:

                    link = listing.get_attribute("href")

                    if (
                        link
                        and link not in visited_links
                    ):

                        links.append(link)

                        visited_links.add(link)

                except:
                    pass

            print(f"Collected {len(links)} links")


            for index, link in enumerate(links):

                try:

                    print(
                        f"{district} | "
                        f"{index+1}/{len(links)}"
                    )

                    driver.get(link)

                    time.sleep(
                        random.uniform(2, 4)
                    )

                    data = extract_school_data()

                    if data:

                        data["District"] = district

                        data["Search Type"] = search_type

                        all_data.append(data)

                        print(data["Hospital Name"])


                    if len(all_data) % 100 == 0:

                        backup_df = pd.DataFrame(
                            all_data
                        )

                        backup_df.drop_duplicates(
                            subset=[
                                "Hospital Name",
                                "Google Maps URL"
                            ],
                            inplace=True
                        )

                        backup_filename = (
                            "karnataka_backup.csv"
                        )

                        backup_df.to_csv(
                            backup_filename,
                            index=False
                        )

                        print("Backup Saved")

                except Exception as e:

                    print(
                        f"School Error: {e}"
                    )

        except Exception as e:

            print(
                f"District Error: {e}"
            )


df = pd.DataFrame(all_data)

df.drop_duplicates(

    subset=[
        "Hospital Name",
        "Google Maps URL"
    ],

    inplace=True
)


filename = (
    "karnataka_hospitals.csv"
)

df.to_csv(
    filename,
    index=False
)

print("\nExtraction Completed")

print(
    f"Total Hospitals Extracted: {len(df)}"
)

print(f"\nSaved File: {filename}")

driver.quit()


Opening Google Maps...
Google Maps Loaded

Searching: hospitals in Bagalkot Karnataka
Search Submitted
Bagalkot | hospitals | Scroll 1 -> 14
Bagalkot | hospitals | Scroll 2 -> 20
Bagalkot | hospitals | Scroll 3 -> 27
Bagalkot | hospitals | Scroll 4 -> 34
Bagalkot | hospitals | Scroll 5 -> 40
Bagalkot | hospitals | Scroll 6 -> 47
Bagalkot | hospitals | Scroll 7 -> 54
Bagalkot | hospitals | Scroll 8 -> 60
Bagalkot | hospitals | Scroll 9 -> 62
Bagalkot | hospitals | Scroll 10 -> 62
Bagalkot | hospitals | Scroll 11 -> 62
Bagalkot | hospitals | Scroll 12 -> 62
Bagalkot | hospitals | Scroll 13 -> 62
Collected 62 links
Bagalkot | 1/62
A V Daddenavar Hospital
Bagalkot | 2/62
Daddenavar Hospital
Bagalkot | 3/62
Mirji Hospital
Bagalkot | 4/62
Dr Guled Hospital
Bagalkot | 5/62
Katti Hospital
Bagalkot | 6/62
Government Hospital
Bagalkot | 7/62
Kerudi multispeciality hospital
Bagalkot | 8/62
Mane Hospital,LifeNu IVF Centre
Bagalkot | 9/62
Shakuntala Multispeciality Hospital
Bagalkot | 10/62
ASKI Su